In [0]:
# NOTEBOOK 1 - PRODUCER
# Run this first and keep it running

%pip install faker pytz

import time
import json
import random
from faker import Faker
from datetime import datetime
import pytz

fake = Faker()

#dbutils.fs.mkdirs("dbfs:/FileStore/manishGautam/transactionStreaming/raw/")

def generate_transaction():
    return {
        "transaction_id": fake.uuid4(),
        "user_id": random.randint(1000, 9999),
        "user_name": fake.name(),
        "email": fake.email(),
        "amount": round(random.uniform(5, 3000), 2),
        "currency": random.choice(["USD", "EUR", "GBP", "JPY"]),
        "merchant": fake.company(),
        "merchant_city": fake.city(),
        "merchant_country": fake.country_code(),
        "category": random.choice(["retail", "food", "travel", "entertainment", "healthcare", "utilities"]),
        "payment_method": random.choice(["credit_card", "debit_card", "mobile_pay", "bank_transfer"]),
        "card_last4": fake.credit_card_number()[-4:],
        "status": random.choices(
            ["approved", "declined", "pending"],
            weights=[78, 15, 7]
        )[0],
        "timestamp": datetime.now().isoformat(),
        "ip_address": fake.ipv4(),
        "device": random.choice(["mobile", "desktop", "tablet", "pos_terminal"])
    }

print("source started — generating transactions every 1 seconds...")


batch_num = 0
while True:
    batch_num += 1
    batch = [generate_transaction() for _ in range(1)]
    
    path = f"/dbfs/FileStore/ManishGautam/medallionArch/orders/2026-03-10/batch_{int(time.time())}_{batch_num}.json"
    with open(path, "w") as f:
        for record in batch:
            f.write(json.dumps(record) + "\n")
    
    print(f"**** Batch {batch_num} written — {len(batch)} transactions at {datetime.now().strftime('%H:%M:%S')} ****")
    time.sleep(1)